[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lanzlagman/intro-python-astro-PUP/blob/main/solutions/02_downloading_datasets_astroquery.ipynb)

> **Solutions copy.** Every **YOUR TURN** cell below is already filled in. Work through `notebooks/` first; read this afterwards.

# Notebook 02: Downloading Datasets with `astroquery`

**Computational and Data-driven Astrophysics: A Practical Python Workshop**
PUP Physics Society · 15 August 2026

---

In Notebook 01 you built a `SkyCoord` pointing at the Pleiades. Here you hand that position
to the European Space Agency and get back **thousands of stars in about ten seconds**.

> **archive → table → filter → save**

Learn it once. Same four steps for Gaia, VizieR, SDSS, MAST and IRSA. The archive changes;
the pattern does not.

**CLASSROOM SWITCH**

If the venue Wi-Fi is struggling, or if 40 people query ESA at once and the
archive throttles us, set this to False and everything below still runs.

## Part 0: Setup

In [1]:
try:
    import astroquery
except ImportError:
    !pip install -q astroquery

import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

TRY_LIVE_QUERY = True

import os
import pandas as pd

RAW = "https://raw.githubusercontent.com/lanzlagman/intro-python-astro-PUP/main/data/"


def load_workshop_data(filename):
    """Local file first, then GitHub. Works identically in Colab and locally."""
    for path in (f"data/{filename}", f"../data/{filename}"):
        if os.path.exists(path):
            print(f"loaded from disk: {path}")
            return pd.read_csv(path)
    print(f"downloading {filename} from GitHub")
    return pd.read_csv(RAW + filename)

---
## Part 1: Choosing a target

The Pleiades (M45) is a **galactic (open) cluster** in Taurus, about **130 pc** away.

Every star in it formed at roughly the same time, from the same gas, at the same distance. Because the cluster is small compared with its distance from Earth, we can treat every member as having the same distance modulus; so any difference between two members comes down to **mass alone**.

That single assumption is why clusters are the best laboratory in stellar astrophysics, and
it is what makes the rest of this workshop work.

In [2]:
# The same SkyCoord from Notebook 01
pleiades = SkyCoord(ra="03h47m00s", dec="+24d07m00s", frame="icrs")
SEARCH_RADIUS = 1.0 * u.deg

print(f"target : Pleiades (M45)")
print(f"centre : ra = {pleiades.ra.deg:.4f} deg, dec = {pleiades.dec.deg:.4f} deg")
print(f"radius : {SEARCH_RADIUS}")
print(f"\nThe cluster is about 130 pc away, so we expect a parallax near "
      f"{1000/130:.2f} mas.")

target : Pleiades (M45)
centre : ra = 56.7500 deg, dec = 24.1167 deg
radius : 1.0 deg

The cluster is about 130 pc away, so we expect a parallax near 7.69 mas.


---
## Part 2: archive → table

Gaia is queried with **ADQL**, which is SQL with two extra geometry functions.
If you have written SQL before, you already know this. If you have not, read it as English:

```
SELECT   these columns
FROM     this catalogue
WHERE    the star falls inside this circle on the sky
```

`CONTAINS(POINT(...), CIRCLE(...))` returns 1 if the point is inside the circle.
That is the whole trick: a **cone search**.

In [3]:
ADQL = f"""
SELECT TOP 8000
    source_id, ra, dec,
    parallax, parallax_error, parallax_over_error,
    pmra, pmdec, pmra_error, pmdec_error,
    phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag, bp_rp,
    ruwe
FROM gaiadr3.gaia_source
WHERE 1 = CONTAINS(
        POINT('ICRS', ra, dec),
        CIRCLE('ICRS', {pleiades.ra.deg}, {pleiades.dec.deg}, {SEARCH_RADIUS.value}))
    AND parallax IS NOT NULL
    AND phot_g_mean_mag < 18
"""
print(ADQL)


SELECT TOP 8000
    source_id, ra, dec,
    parallax, parallax_error, parallax_over_error,
    pmra, pmdec, pmra_error, pmdec_error,
    phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag, bp_rp,
    ruwe
FROM gaiadr3.gaia_source
WHERE 1 = CONTAINS(
        POINT('ICRS', ra, dec),
        CIRCLE('ICRS', 56.74999999999999, 24.116666666666667, 1.0))
    AND parallax IS NOT NULL
    AND phot_g_mean_mag < 18



### The `try / except` habit

Never let a network call be the thing that kills a live demo: or a thesis run at 2 a.m. Every cell in this repository that touches the internet has a fallback.

In [4]:
def fetch_gaia():
    """Live Gaia query, with a cached fallback. Returns a pandas DataFrame."""
    if not TRY_LIVE_QUERY:
        print("TRY_LIVE_QUERY is False -> using cached data")
        return load_workshop_data("gaia_pleiades.csv")

    try:
        from astroquery.gaia import Gaia
        job = Gaia.launch_job_async(ADQL)
        df = job.get_results().to_pandas()
        print(f"live query OK: {len(df)} rows from gaiadr3.gaia_source")
        return df
    except Exception as err:
        print(f"live query failed ({type(err).__name__}) -> falling back to cache")
        return load_workshop_data("gaia_pleiades.csv")


gaia = fetch_gaia()
print(f"\nshape: {gaia.shape}")
gaia.head()

INFO: Query finished. [astroquery.utils.tap.core]
live query OK: 7797 rows from gaiadr3.gaia_source

shape: (7797, 15)


,source_id,ra,dec,parallax,parallax_error,parallax_over_error,pmra,pmdec,pmra_error,pmdec_error,phot_g_mean_mag,phot_bp_mean_mag,phot_rp_mean_mag,bp_rp,ruwe
0,64879402312818944,56.213581,23.268724,7.394625,0.018698,395.469757,20.545926,-44.300316,0.022893,0.016450,8.886077,9.084315,8.536390,0.547925,0.980512
1,64910149982598144,56.976257,23.175415,0.393595,0.100163,3.929553,1.732303,-2.428907,0.128292,0.085494,17.311483,17.750784,16.723562,1.027222,1.045113
2,64911971048493184,56.815338,23.144629,0.305246,0.098291,3.105514,4.922191,-0.236094,0.120585,0.081471,17.365173,17.831347,16.774443,1.056904,0.891074
3,64912112784703488,56.842837,23.172631,1.073798,0.024692,43.488037,1.053956,-4.633143,0.030179,0.020669,14.571305,15.044904,13.939600,1.105304,0.982329
4,64912142847184768,56.804243,23.142868,0.220597,0.101313,2.177387,0.810286,-1.113088,0.137248,0.080833,17.149969,17.511322,16.627186,0.884136,1.076247


In [5]:
# Step 2 complete: we have a table. Look at what an archive actually gives you.
print("columns:", list(gaia.columns))
gaia[["parallax", "pmra", "pmdec", "phot_g_mean_mag", "bp_rp", "ruwe"]].describe().round(2)

columns: ['source_id', 'ra', 'dec', 'parallax', 'parallax_error', 'parallax_over_error', 'pmra', 'pmdec', 'pmra_error', 'pmdec_error', 'phot_g_mean_mag', 'phot_bp_mean_mag', 'phot_rp_mean_mag', 'bp_rp', 'ruwe']


,parallax,pmra,pmdec,phot_g_mean_mag,bp_rp,ruwe
count,7797.00,7797.00,7797.00,7797.00,7751.00,7797.00
mean,1.40,4.47,-7.78,16.11,1.39,1.22
std,2.04,11.48,13.45,1.82,0.52,1.27
min,-2.60,-67.06,-174.78,3.62,-0.11,0.75
25%,0.37,-0.15,-8.35,15.41,1.05,0.98
50%,0.71,2.01,-3.59,16.66,1.24,1.02
75%,1.35,6.07,-1.04,17.41,1.55,1.07
max,41.46,331.27,123.16,18.00,3.67,38.56


> **Read the `describe()` output before you plot anything.** The parallax column runs from
> near zero to several mas. Most of those stars are nowhere near the Pleiades; they are
> background objects that merely lie along the same line of sight. A cone search selects a
> *direction*, not a *cluster*. Separating the two is what Notebooks 04 and 05 are about.

---
## Part 3: filter

The classic observational H–R diagrams were built from stars whose parallaxes were measured
to better than 20%, a **quality cut** applied by hand to make a figure. We apply the identical
physical criterion in one line of Python.

Parallax good to 20% means $\sigma_\varpi/\varpi < 0.2$, i.e. $\varpi/\sigma_\varpi > 5$.
Gaia ships that ratio precomputed as `parallax_over_error`.

**`ruwe`** (Renormalised Unit Weight Error) is a Gaia-era diagnostic with no textbook
counterpart. Above ~1.4 the single-star model fitted badly, usually an unresolved binary or a
crowded field. Cutting on it is standard practice.

In [6]:
if "parallax_over_error" not in gaia.columns:
    gaia["parallax_over_error"] = gaia["parallax"] / gaia["parallax_error"]

# The "better than 20%" parallax criterion, written as code:
good = gaia[(gaia["parallax_over_error"] > 5) & (gaia["ruwe"] < 1.4)].copy()

print(f"raw cone      : {len(gaia):5d} stars")
print(f"after quality : {len(good):5d} stars   ({100*len(good)/len(gaia):.0f}% kept)")
print(f"\nThe classic Hipparcos H-R diagram used ~3700 stars, assembled by hand.")
print(f"We just applied the same cut to {len(good)} stars in one line, from one cluster.")

raw cone      :  7797 stars
after quality :  5180 stars   (66% kept)

The classic Hipparcos H-R diagram used ~3700 stars, assembled by hand.
We just applied the same cut to 5180 stars in one line, from one cluster.


### ✏️ YOUR TURN

The Pleiades sits near **7.4 mas**. Add a parallax cut that keeps only stars plausibly at the
cluster distance, then see how many survive.

In [7]:
# YOUR TURN: keep stars with parallax between 6 and 9 mas
# Hint: good[(good["parallax"] > ...) & (good["parallax"] < ...)]
near = good[(good["parallax"] > 6) & (good["parallax"] < 9)]

# ---- check ----
if near is None:
    print("Fill in the line above, then re-run.")
else:
    print(f"stars near the cluster distance: {len(near)}")
    print(f"median parallax : {near['parallax'].median():.2f} mas")
    print(f"implied distance: {1000/near['parallax'].median():.0f} pc")
    print()
    print("The textbook value is 130 pc. You should get something nearer 136 pc.")
    print("That gap is real and it is not your error: the book went to press in 2007,")
    print("before Gaia. The Pleiades distance was genuinely contested for years -")
    print("Hipparcos said ~120 pc and everyone else said ~134 pc. Gaia settled it.")
    print("Textbook numbers have dates on them.")

stars near the cluster distance: 433
median parallax : 7.36 mas
implied distance: 136 pc

The textbook value is 130 pc. You should get something nearer 136 pc.
That gap is real and it is not your error: the book went to press in 2007,
before Gaia. The Pleiades distance was genuinely contested for years -
Hipparcos said ~120 pc and everyone else said ~134 pc. Gaia settled it.
Textbook numbers have dates on them.


---
## Part 4: The same pattern, a different archive

Gaia gives us *measurements*. To check our work later we also want somebody else's
*conclusions*: a published membership list. That lives in **VizieR**, a different archive
with a completely different backend.

Watch how little the code changes.

In [8]:
def fetch_published_members():
    """A published Pleiades membership list. Same four-step pattern, new archive."""
    if not TRY_LIVE_QUERY:
        return load_workshop_data("pleiades_members_published.csv")
    try:
        from astroquery.vizier import Vizier
        v = Vizier(columns=["Source"], row_limit=-1)
        # Catalogue J/A+A/628/A66 = Gaia DR2 open cluster members (Cantat-Gaudin+ 2019)
        result = v.query_constraints(catalog="J/A+A/628/A66", Cluster="Melotte_22")
        df = result[0].to_pandas().rename(columns={"Source": "source_id"})
        print(f"live VizieR query OK: {len(df)} published members")
        return df
    except Exception as err:
        print(f"VizieR unavailable ({type(err).__name__}) -> using cache")
        return load_workshop_data("pleiades_members_published.csv")

published = fetch_published_members()
print(f"published member list: {len(published)} stars")
published.head()

live VizieR query OK: 3162 published members
published member list: 3162 stars


,source_id
0,455075360092433920
1,455037048980464896
2,455026156947855872
3,355242590505237888
4,355343947436467200


> **That is the whole lesson of this notebook.** Two archives, two query languages
> underneath, and from your side the code is the same shape both times: call a function, get a
> table, filter it, keep it.
>
> You do not need to memorise ADQL or the VizieR naming scheme. You need to know the pattern
> exists, and where the documentation is.

### ⚠️ One important exception: gravitational waves

Gravitational-wave strain data does **not** come through `astroquery`. LIGO/Virgo open
data lives at GWOSC and is accessed with the `gwpy` package:

```python
from gwpy.timeseries import TimeSeries
strain = TimeSeries.fetch_open_data("H1", 1126259446, 1126259478)   # GW150914
```

Different library, different idea: a continuous time series rather than a catalogue of
objects. Mentioning it now so that when you meet it in the advanced track it looks like a
deliberate exception rather than the pattern mysteriously breaking.

---
## Part 5: save

Saving is not bookkeeping. It is what makes your result **reproducible**: you can rerun
tomorrow's analysis on exactly today's rows, whether or not the archive is up, and whether
or not the catalogue has been revised in the meantime.

In [9]:
import os
os.makedirs("data", exist_ok=True)

good.to_csv("data/pleiades_query_result.csv", index=False)
print(f"saved {len(good)} rows -> data/pleiades_query_result.csv")

# Quick preview of what Notebook 04 will plot
print("\nWhat we are handing to Notebook 04:")
print(f"  {len(good)} stars")
print(f"  colour  (bp_rp)           : {good['bp_rp'].min():.2f} to {good['bp_rp'].max():.2f}")
print(f"  brightness (phot_g_mean_mag): {good['phot_g_mean_mag'].min():.1f} to "
      f"{good['phot_g_mean_mag'].max():.1f}")
print(f"  proper motion pmra        : {good['pmra'].min():.0f} to {good['pmra'].max():.0f} mas/yr")

saved 5180 rows -> data/pleiades_query_result.csv

What we are handing to Notebook 04:
  5180 stars
  colour  (bp_rp)           : -0.08 to 3.67
  brightness (phot_g_mean_mag): 5.2 to 18.0
  proper motion pmra        : -67 to 331 mas/yr


---
## What you built, and where it goes

| You wrote | Next used in |
|---|---|
| The cone search / ADQL pattern | any archive, for the rest of your career |
| `try / except` around every network call | every notebook in the advanced track |
| Quality cuts on parallax and `ruwe` | **NB04** and **NB05** |
| `data/pleiades_query_result.csv` | **NB04** plots it, **NB05** clusters it |
| A published member list | **NB05:** the ground truth we score against |

**Notebook 03** steps away from downloaded data entirely. There is a second way to get data in
astrophysics: **generate it from the physics**.

---
### Sources

Cluster and H–R diagram background follows Carroll & Ostlie, *An Introduction to Modern
Astrophysics*, 2nd ed.: §1.3 (Positions on the Celestial Sphere), §3.6 (The Color Index),
§8.2 (The Hertzsprung–Russell Diagram), §13.3 (Stellar Clusters).

### If you want to go deeper
- Gaia archive & ADQL cookbook · <https://gea.esac.esa.int/archive/>
- Learn Astropy: *Data I/O* and *Working with catalogues* · <https://learn.astropy.org/>
- Pasha & Agostino, *Data I/O: Working with SDSS Data* · <https://prappleizer.github.io/>